# 按句子和父子片段补充上下文

小片段容易找准，却可能没有足够内容回答。本页把 Sentence Window、Small-to-Big 和 AutoMerging 放到《南瓜书》的真实检索中：先取实际命中的小片段，再按原文中的句子或章节关系补回上下文。

Sentence Window 使用同页内被切断的过拟合与欠拟合解释；Small-to-Big 和 AutoMerging 分别使用线性回归、信念传播、高斯混合和 Gibbs/MH 的独立跨页案例。这样三种方法各自面对适合自己的问题，不要求句子窗口解决跨页缺失。

In [1]:
import json
import re
import sys
from pathlib import Path

def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / 'data' / 'dataset/manifest.json').is_file():
            return folder
    raise FileNotFoundError('没有找到教程数据目录，请从本节所在目录运行。')

course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

from common.eval_utils import emit_tutorial_audit
from common.eval_utils import (
    build_bm25_chunk_search,
    build_bm25_search,
    load_query_catalog,
    load_pdf_pages,
    make_fixed_chunks,
)
from common.nontraining_utils import load_annotation

case_data = load_query_catalog()
cases = {item['id']: item for item in case_data}
pages = load_pdf_pages()
page_by_number = {page['page']: page for page in pages}
page_search = build_bm25_search(pages)
fixed_chunks = make_fixed_chunks(pages, chunk_size=180, overlap=0)
chunk_search = build_bm25_chunk_search(fixed_chunks)
short_chunks = make_fixed_chunks(pages, chunk_size=40, overlap=0)
short_chunk_search = build_bm25_chunk_search(short_chunks)

def target_rank(results, expected_pages):
    expected = set(expected_pages)
    return next((index for index, item in enumerate(results, 1) if expected.intersection(item.pages if hasattr(item, 'pages') else [item.page])), None)

def sentence_parts(text):
    return [part.strip() for part in re.split(r'(?<=[。！？.!?])\s+', text) if part.strip()]

def sentence_spans(text):
    spans = []
    cursor = 0
    for sentence in sentence_parts(text):
        start = text.find(sentence, cursor)
        if start < 0:
            continue
        spans.append((start, start + len(sentence), sentence))
        cursor = start + len(sentence)
    return spans

def sentence_window_from_hit(hit, radius=1):
    # 以真实检索命中的短片段在原页中的位置为锚点，不用答案短语挑选命中。
    page_text = page_by_number[hit.pages[0]]['text']
    chunk_text = hit.text.strip()
    start = page_text.find(chunk_text)
    if start < 0:
        terms = re.findall(r'[一-鿿A-Za-z0-9]{2,}', chunk_text)
        positions = [page_text.find(term) for term in terms if page_text.find(term) >= 0]
        start = min(positions, default=0)
    end = start + len(chunk_text)
    spans = sentence_spans(page_text)
    hit_index = next((index for index, (left, right, _) in enumerate(spans)
                      if left < end and right > start), 0)
    return [sentence for _, _, sentence in spans[max(0, hit_index - radius):
                                               min(len(spans), hit_index + radius + 1)]]

CANDIDATE_K = 5

def text_cost(items):
    return sum(len(item.text if hasattr(item, 'text') else str(item)) for item in items)

def page_text_cost(page_numbers):
    return sum(len(page_by_number[page]['text']) for page in sorted(set(page_numbers)))

# 这些父范围只来自当前 PDF 的章节边界；它们不是用来提前选择答案页的。
parent_ranges = {
    'linear_regression_parent': range(33, 35),
    'belief_propagation_parent': range(182, 184),
}
auto_parent_ranges = {
    'gmm_update_parent': range(116, 119),
    'gibbs_mh_parent': range(184, 186),
}

def parent_for_page(page_number):
    for parent, page_range in parent_ranges.items():
        if page_number in page_range:
            return parent
    return f'page_{page_number}'

def coverage(page_numbers, expected_pages):
    expected = set(expected_pages)
    return len(expected.intersection(page_numbers)), len(expected), sorted(expected.intersection(page_numbers))

def first_required_rank(results, expected_pages):
    expected = set(expected_pages)
    for index, item in enumerate(results, 1):
        if expected.intersection(item.pages):
            return index
    return None

def pages_for_parent(ranges, parent):
    return [page for page in ranges[parent] if page in page_by_number]

def saved_case_result(case, before_hits, after_pages, after_rank=1, method=None, role=None, check_purpose=None):
    annotation = load_annotation(case['id'])
    before_pages = sorted({page for hit in before_hits for page in hit.pages})
    before_found = coverage(before_pages, annotation['expected_pages'])
    after_found = coverage(after_pages, annotation['expected_pages'])
    return {
        'case_id': case['id'],
        **({'method': method} if method is not None else {}),
        **({'role': role} if role is not None else {}),
        **({'check_purpose': check_purpose} if check_purpose is not None else {}),
        'before': {
            'required_page_coverage': before_found[0] / before_found[1] if before_found[1] else 0,
            'first_required_rank': first_required_rank(before_hits, annotation['expected_pages']),
            'pages': before_pages,
        },
        'after': {
            'required_page_coverage': after_found[0] / after_found[1] if after_found[1] else 0,
            'first_required_rank': after_rank,
            'pages': sorted({int(page) for page in after_pages}),
        },
    }

# Small-to-Big 使用 120 字 child 定位，命中后只回填当前 PDF 中的父范围。
small_children = make_fixed_chunks(pages, chunk_size=120, overlap=0)
small_search = build_bm25_chunk_search(small_children)
small_main = cases['linear_regression_closed_form_vectorization']
small_main_before = small_search(small_main['query'], top_k=CANDIDATE_K)
small_main_anchor = small_main_before[0]
small_main_parent = parent_for_page(small_main_anchor.pages[0])
small_main_after_pages = pages_for_parent(parent_ranges, small_main_parent)
assert small_main_anchor.pages == [34] and small_main_after_pages == [33, 34]
print('Small-to-Big：线性回归闭式解问题，候选', len(small_main_before), '条；补回父片段后返回页', small_main_after_pages, '；文字量', text_cost(small_main_before), '→', page_text_cost(small_main_after_pages), '字')
emit_tutorial_audit(saved_case_result(small_main, small_main_before, small_main_after_pages, method='Small-to-Big（父子片段）', role='main'))

small_check = cases['belief_propagation_two_passes']
small_check_before = small_search(small_check['query'], top_k=CANDIDATE_K)
small_check_parent = parent_for_page(small_check_before[0].pages[0])
small_check_after_pages = pages_for_parent(parent_ranges, small_check_parent)
assert small_check_before[0].pages == [183] and small_check_after_pages == [182, 183]
print('Small-to-Big 对照：无向图消息传递问题，候选', len(small_check_before), '条；补回父片段后返回页', small_check_after_pages, '；文字量', text_cost(small_check_before), '→', page_text_cost(small_check_after_pages), '字')
emit_tutorial_audit(saved_case_result(small_check, small_check_before, small_check_after_pages, method='Small-to-Big（父子片段）', role='check', check_purpose='再次改善'))

# AutoMerging 用 600 字 child；只有同一父范围至少命中两个 child 且命中比例达到阈值才抬升。
def make_parent_children(ranges, chunk_size=600):
    rows = []
    for parent, page_range in ranges.items():
        for page in page_range:
            text = page_by_number[page]['text']
            for index, start in enumerate(range(0, len(text), chunk_size), 1):
                rows.append({'chunk_id': f'{parent}_child_{page}_{index}', 'pages': [page], 'text': text[start:start + chunk_size]})
    return rows

auto_children = make_parent_children(auto_parent_ranges)
auto_search = build_bm25_chunk_search(auto_children)
def auto_merge(query, top_k=CANDIDATE_K, threshold=0.25):
    hits = auto_search(query, top_k=top_k)
    for parent, page_range in auto_parent_ranges.items():
        parent_children = [row for row in auto_children if row['pages'][0] in page_range]
        parent_ids = {row['chunk_id'] for row in parent_children}
        parent_hits = [hit for hit in hits if hit.chunk_id in parent_ids]
        ratio = len({hit.chunk_id for hit in parent_hits}) / max(1, len(parent_children))
        if len(parent_hits) >= 2 and ratio > threshold:
            return parent_hits, parent, ratio
    raise AssertionError('没有达到 AutoMerging 的同父命中条件')

auto_main = cases['gmm_parameter_update_context']
auto_main_before = auto_search(auto_main['query'], top_k=CANDIDATE_K)
auto_main_hits, auto_main_parent, auto_main_ratio = auto_merge(auto_main['query'], top_k=CANDIDATE_K)
auto_main_after_pages = pages_for_parent(auto_parent_ranges, auto_main_parent)
assert auto_main_before[0].pages == [117] and auto_main_parent == 'gmm_update_parent' and auto_main_after_pages == [116, 117, 118]
print('AutoMerging：高斯混合模型参数更新问题，候选', len(auto_main_before), '条、同父命中', len(auto_main_hits), '条；补回后返回页', auto_main_after_pages, '；文字量', text_cost(auto_main_before), '→', page_text_cost(auto_main_after_pages), '字；命中比例', round(auto_main_ratio, 3))
emit_tutorial_audit(saved_case_result(auto_main, auto_main_before, auto_main_after_pages, method='AutoMerging（按命中比例合并）', role='main'))

auto_check = cases['gibbs_mh_acceptance_context']
auto_check_before = auto_search(auto_check['query'], top_k=CANDIDATE_K)
auto_check_hits, auto_check_parent, auto_check_ratio = auto_merge(auto_check['query'], top_k=CANDIDATE_K)
auto_check_after_pages = pages_for_parent(auto_parent_ranges, auto_check_parent)
assert auto_check_before[0].pages == [185] and auto_check_parent == 'gibbs_mh_parent' and auto_check_after_pages == [184, 185]
print('AutoMerging 对照：吉布斯采样接受概率问题，候选', len(auto_check_before), '条、同父命中', len(auto_check_hits), '条；补回后返回页', auto_check_after_pages, '；文字量', text_cost(auto_check_before), '→', page_text_cost(auto_check_after_pages), '字；命中比例', round(auto_check_ratio, 3))
emit_tutorial_audit(saved_case_result(auto_check, auto_check_before, auto_check_after_pages, method='AutoMerging（按命中比例合并）', role='check', check_purpose='确认没有改坏'))

# Sentence Window 的独立问题：先取真实检索第一条，再按它在原页中的句子位置补回相邻句。
sentence_case = cases['overfitting_underfitting']
sentence_anchor_results = short_chunk_search(sentence_case['query'], top_k=CANDIDATE_K)
sentence_hit = sentence_anchor_results[0]
sentence_after = sentence_window_from_hit(sentence_hit, radius=1)
sentence_before_text = sentence_hit.text
sentence_after_text = ' '.join(sentence_after)
print('\nSentence Window：问题：', sentence_case['query'])
print('改动前原文：', sentence_before_text)
print('改动后原文：', sentence_after_text)
print('命中页/候选数/文字量：', sentence_hit.pages[0], '/', len(sentence_anchor_results), '/', len(sentence_before_text), '→', len(sentence_after_text), '字')
sentence_annotation = load_annotation(sentence_case['id'])
sentence_items = []
for evidence in sentence_annotation.get('essential_evidence_spans', []):
    for span in evidence.get('spans', []):
        sentence_items.extend(
            re.sub(r'\s+', '', part) for part in re.split(r'[，；。]', span.get('quote', '')) if len(re.sub(r'\s+', '', part)) >= 6
        )
def explanation_coverage(text):
    normalized = re.sub(r'\s+', '', text)
    return sum(item in normalized for item in sentence_items)
explanation_before = explanation_coverage(sentence_before_text)
explanation_after = explanation_coverage(sentence_after_text)
print('必要解释项覆盖：', explanation_before, '→', explanation_after, '；结论：相邻句补回了同一段解释。')

sentence_check = cases['business_model_selection']
sentence_check_results = short_chunk_search(sentence_check['query'], top_k=CANDIDATE_K)
sentence_check_hit = sentence_check_results[0]
sentence_check_after = sentence_window_from_hit(sentence_check_hit, radius=1)
print('\nSentence Window 对照：问题：', sentence_check['query'])
print('改动前原文：', sentence_check_hit.text)
print('改动后原文：', ' '.join(sentence_check_after))
print('命中页/候选数/文字量：', sentence_check_hit.pages[0], '/', len(sentence_check_results), '/', len(sentence_check_hit.text), '→', len(' '.join(sentence_check_after)), '字')
assert sentence_check_hit.pages == [18]
assert any('业务场景' in text for text in sentence_check_after)

print('本页三个方法都已保存可阅读的前后对照；Sentence Window 保留自己的局部上下文实验。')

Small-to-Big：线性回归闭式解问题，候选 5 条；补回父片段后返回页 [33, 34] ；文字量 598 → 2408 字


Small-to-Big 对照：无向图消息传递问题，候选 5 条；补回父片段后返回页 [182, 183] ；文字量 598 → 2919 字


AutoMerging：高斯混合模型参数更新问题，候选 5 条、同父命中 3 条；补回后返回页 [116, 117, 118] ；文字量 3000 → 3568 字；命中比例 0.375


AutoMerging 对照：吉布斯采样接受概率问题，候选 5 条、同父命中 3 条；补回后返回页 [184, 185] ；文字量 2931 → 3765 字；命中比例 0.429



Sentence Window：问题： 过拟合和欠拟合分别说明模型的学习能力相对数据过强还是过弱？
改动前原文： 过拟合是由于模型的学习能力相对于数据来说过于强大，反过来说，欠拟合是因为模型的学
改动后原文： 错误率和精度很容易理解，而且很明显是针对分类问题的。误差的概念更适用于回归问题，但是，根 据“西瓜书”第12 章的式(12.1) 和式(12.2) 的定义可以看出，在分类问题中也会使用误差的概念，此时 的“差异”指的是学习器的实际预测输出的类别与样本真实的类别是否一致，若一致则“差异”为0，若 不一致则“差异”为1，训练误差是在训练集上差异的平均值，而泛化误差则是在新样本（训练集中未出 现过的样本）上差异的平均值。 过拟合是由于模型的学习能力相对于数据来说过于强大，反过来说，欠拟合是因为模型的学习能力相 对于数据来说过于低下。暂且抛开“没有免费的午餐”定理不谈，例如对于“西瓜书”第1 章图1.4 中的 训练样本（黑点）来说，用类似于抛物线的曲线A 去拟合则较为合理，而比较崎岖的曲线B 相对于训练 样本来说学习能力过于强大，但若仅用一条直线去训练则相对于训练样本来说直线的学习能力过于低下。 2.2 评估方法 本节介绍了3 种模型评估方法：留出法、交叉验证法、自助法。留出法由于操作简单，因此最常用； 交叉验证法常用于对比同一算法的不同参数配置之间的效果，以及对比不同算法之间的效果；自助法常用 于集成学习（详见“西瓜书”第8 章的8.2 节和8.3 节）产生基分类器。留出法和自助法简单易懂，在此 不再赘述，下面举例说明交叉验证法的常用方式。
命中页/候选数/文字量： 18 / 5 / 40 → 576 字
必要解释项覆盖： 1 → 2 ；结论：相邻句补回了同一段解释。



Sentence Window 对照：问题： 如何判断模型好坏并挑选适合业务场景的模型？
改动前原文： 是如何评估模型的优劣和选择最适合自己业务场景的模型。 由于“模型评估与选择”是在
改动后原文： 第2 章 模型评估与选择 如“西瓜书”前言所述，本章仍属于机器学习基础知识，如果说第1 章介绍了什么是机器学习及机 器学习的相关数学符号，那么本章则进一步介绍机器学习的相关概念。具体来说，介绍内容正如本章名称 “模型评估与选择”所述，讲述的是如何评估模型的优劣和选择最适合自己业务场景的模型。 由于“模型评估与选择”是在模型产出以后进行的下游工作，要想完全吸收本章内容需要读者对模型 有一些基本的认知，因此零基础的读者直接看本章会很吃力，实属正常，在此建议零基础的读者可以简单 泛读本章，仅看能看懂的部分即可，或者直接跳过本章从第3 章开始看，直至看完第6 章以后再回头来看 本章便会轻松许多。
命中页/候选数/文字量： 18 / 5 / 40 → 297 字
本页三个方法都已保存可阅读的前后对照；Sentence Window 保留自己的局部上下文实验。


Sentence Window 的独立问题中，40 字命中片段只保留“学习能力过于强大”，补回完整句子后同时得到过拟合“过于强大”和欠拟合“过于低下”。它只在相邻句内扩展，不能替代跨页父子关系。

Small-to-Big 的线性回归案例从第 34 页小片段回填到第 33～34 页所属大段；AutoMerging 的高斯混合案例需要同一所属大段命中多个小片段，才回填第 116～118 页。Gibbs/MH 复查也保留第 184～185 页的原文关系。这里的 AutoMerging 检索索引只覆盖两个手工选定的父范围，共五页（第 116～118 页和第 184～185 页），并不是整本书的索引或全书实验。所属大段或句子边界建立错误时，扩展会增加无关文字，因此应同时记录返回字符数，并在实际回答中逐条回到原文核对。



## 原理、适用条件与当前案例

Sentence Window（句子窗口）是在检索命中句的前后补回固定数量的句子；它适合连续叙述中“定义—解释—例子”被切开的情况。窗口应在索引时保存句子顺序，检索时按句子 id 去重后再拼接；窗口过大时会引入噪声，跨段或跨页证据也补不回来。

Small-to-Big（父子片段）把小片段用于定位、把所属父片段用于回答。父片段应有清楚的边界；如果父片段本身没有答案，回填只会增加文字。AutoMerging（自动合并）则先取叶子片段，统计同一父片段下命中的子片段比例，只有达到阈值才抬升父片段。它不是“命中一个子片段就返回整段”，阈值越低越完整但越容易带来无关内容。

本页用“过拟合和欠拟合分别说明模型的学习能力相对数据过强还是过弱？”与“如何判断模型好坏并挑选适合业务场景的模型？”观察 Sentence Window；用“线性回归怎样把一元闭式解写成去均值向量形式，并把多元最小二乘目标写成矩阵形式？”与“无向图模型的消息怎样沿边传递得到边际分布？为什么信念传播先从叶结点到根结点，再从根结点到叶结点？”观察 Small-to-Big；用“高斯混合模型 EM 算法如何用 γji 更新均值、协方差矩阵和混合系数，什么时候停止？”与“候选样本与当前样本除第 i 个变量外其余变量相同，为什么吉布斯采样接受概率为 1？”观察 AutoMerging。三种方法的差异要看实际送入回答模型的文字和必要页覆盖，不只看最终分数。


In [2]:
# 实现要点：只展示上下文补回逻辑，直接复用本页已经保存的索引。
def sentence_window_ids(hit_ids, radius, n_sentences):
    ids = set()
    for hit in hit_ids:
        ids.update(range(max(0, hit - radius), min(n_sentences, hit + radius + 1)))
    return sorted(ids)

def parent_context(child_hits, child_to_parent, parent_texts, max_parents=3):
    parent_ids = []
    for child_id in child_hits:
        parent_id = child_to_parent[child_id]
        if parent_id not in parent_ids:
            parent_ids.append(parent_id)
    return "\n\n".join(parent_texts[i] for i in parent_ids[:max_parents])

def auto_merge_context(child_hits, child_to_parent, children_by_parent, child_texts, parent_texts, threshold=0.5):
    """命中密度超过 threshold 才以父片段替换子片段；其余保留子片段。"""
    hits = set(child_hits); parts = []; merged = set()
    for parent_id, children in children_by_parent.items():
        ratio = len(hits.intersection(children)) / max(1, len(children))
        if ratio > threshold:
            parts.append(parent_texts[parent_id]); merged.update(children)
    parts.extend(child_texts[i] for i in child_hits if i not in merged)
    return "\n\n".join(parts)

# 当前实测对应关系：Sentence Window=overfitting_underfitting/business_model_selection；父子片段案例见上方代码格。


## 三种方法有什么区别

改动前结果已经命中相关位置时，失败往往不是“库里没有答案”，而是命中片段太短、句子在边界处被截断，或同一段的证据分散在多个小片段里。本页只处理这个问题：先检索，再整理送给回答模型的文字；它不改写问题，也不负责多轮规划。

### Sentence Window（句子窗口）

先按句子建立索引并保存顺序。检索命中一句后，取它前后几句，去重并按原文顺序拼接，再限制总字数。这样返回的不再是孤立的一句话，而是一小段连续解释。

它适合定义—解释—例子就在相邻句中的教材正文；不适合答案跨页、跨段，或相邻句主题变化很快的日志。窗口越大，遗漏的风险越小，但噪声和上下文成本也一起增加。

过拟合与欠拟合问题中，40 字的小片段只有“学习能力过于强大”，补回完整句后才出现“过于低下”。另一个业务场景问题在补充前后都能找到正确内容。句子窗口只能补相邻句，不能解决跨页缺失。

### Small-to-Big（父子片段）

建立索引时同时保存小片段和它所属的大段。检索仍在小片段上进行，命中后再返回完整大段：小片段负责定位，大段负责提供回答所需的上下文。如果大段边界划错，或者大段本身没有答案，返回更多文字也没有帮助。

Small-to-Big 更适合“定义、条件、例子和结论分散在同一段”的教材或技术文档，不适合层级不可靠的聊天记录和日志。实现时要给所属大段数量、字符数设上限，否则一个命中就可能把大量无关文字送入生成模型。

### AutoMerging（自动合并）

AutoMerging 先返回叶子小片段，再统计同一个所属大段下命中的小片段比例：

```text
hit_ratio = 命中的小片段数 / 该所属大段的小片段总数
若 hit_ratio > simple_ratio_thresh，则用所属大段替换这些小片段；否则保留小片段。
```

它还可以补齐相邻片段之间的缺口，再重复一次“补缺口 → 按比例合并”。阈值较低时更容易得到完整所属大段，但也更容易引入噪声；阈值较高时更谨慎，却可能留下碎片。它不是“命中一个小片段就无条件返回整段”，这正是它和固定 Small-to-Big 的差别。

Small-to-Big 在两个问题中分别补回第 33～34 页和第 182～183 页。AutoMerging 则要求同一大段中有多个小片段被命中，两个问题分别补回第 116～118 页和第 184～185 页。这些结果来自当前 PDF 和 BM25 检索，只说明这几道题上的变化。

### 横向比较与限制

| 方法 | 检索对象 | 何时扩展 | 主要风险 | 当前证据 |
| --- | --- | --- | --- | --- |
| Sentence Window | 句子 | 命中句周围固定窗口 | 邻句噪声、跨页无效 | 过拟合与业务场景问题 |
| Small-to-Big | 小片段 | 命中后固定回填所属大段 | 所属大段过大或本身不含答案 | 线性回归和信念传播案例各自回填相邻所属大段 |
| AutoMerging | 叶子小片段 | 同一所属大段的命中密度超过阈值 | 阈值依语料变化，合并过度 | 高斯混合和 Gibbs/MH 案例各自要求多个小片段命中 |

读结果时同时看必要页面覆盖、送入回答的字符数和实际补回文本。父子关系或句子边界建立错时，任何扩展都可能把错误放大；因此“上下文更长”不等于“答案更正确”。长文先整体编码再分块是索引阶段的另一条路线，见[长文先整体编码再分块](长文先整体编码再分块.ipynb)。

后面的代码分别完成三件事：按句子顺序取邻近内容、从小片段找到所属大段、计算同一大段中有多少小片段被命中。它只整理检索结果，不会重建向量库。

In [3]:
from common.eval_utils import emit_tutorial_audit

# Sentence Window 和父子片段使用同一份保存契约；页码来自实际结果。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        values = item.pages if hasattr(item, 'pages') else [item.page]
        for page in values:
            page = int(page)
            if page not in pages:
                pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(method, role, case_id, before_items, after_items, purpose=None, comparison=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': method, 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    if comparison:
        payload['comparison'] = comparison
    emit_tutorial_audit(payload)

sentence_after_evidence = [type(sentence_hit)(sentence_hit.chunk_id, sentence_hit.pages, '\n'.join(sentence_after), sentence_hit.score)]
_emit('补回相邻句（Sentence Window）', 'main', 'overfitting_underfitting',
      sentence_anchor_results, sentence_after_evidence,
      comparison={'name': '必要解释项覆盖', 'before': explanation_before,
                  'after': explanation_after, 'higher_is_better': True})
_emit('补回相邻句（Sentence Window）', 'check', 'business_model_selection',
      [sentence_check_hit], [type(sentence_check_hit)(sentence_check_hit.chunk_id, sentence_check_hit.pages, '\n'.join(sentence_check_after), sentence_check_hit.score)],
      '确认没有改坏')
